##**Mounting the Google Drive and Installing libraries**

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# !pip install thefuzz python-Levenshtein

In [3]:
import numpy as np
import pandas as pd
import math

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from thefuzz import process

from scipy.optimize import linprog

import matplotlib.pyplot as plt

import pickle
import os
from collections import defaultdict
import json
import pprint
from tabulate import tabulate

#**Model Training**

**Loading the Delivery Data for the (2022-2025) Seasons**

In [4]:
DATASET_DIR = 'Datasets/IPL DELIVERIES/'

data = pd.read_csv(DATASET_DIR + 'ipl_2022_deliveries.csv')
for i in [2023,2024,2025]:
  data = pd.concat([data, pd.read_csv(DATASET_DIR + f'ipl_{i}_deliveries.csv')])

df_train = pd.DataFrame(data)

df_train['date'] = pd.to_datetime(df_train['date'], format='%b %d, %Y')

X_train = df_train[['season', 'date', 'venue', 'batting_team', 'bowling_team', 'over', 'striker', 'bowler', 'runs_of_bat', 'extras', 'wide', 'legbyes', 'byes', 'noballs', 'wicket_type', 'player_dismissed', 'fielder']]

**Making the Player Names within the Dataframe Consistent**

In [5]:
# Function to find similar names within a single dataframe
# using the "process" method

def find_similar_names(df, threshold=90):
  name_columns = ['striker', 'bowler', 'fielder', 'player_dismissed']
  all_names = pd.Series(dtype=str)

  # Gather every unique name across all columns
  for col in name_columns:
    if col in df.columns:
      all_names = pd.concat([all_names, df[col].dropna()])

  unique_names = all_names.unique().tolist()

  matches = {}
  seen_pairs = set() # Tracks pairs we've already matched to prevent reverse duplicates

  for name in unique_names:
    # Skip if the first argument (the key) contains a slash
    # These slashes appear between the names of different players in a
    # single entry, whenever they are involved in a combined fielding
    # effort
    if '/' in name:
      continue
    # Skip if the key contains "(sub)"
    # These are substitution instances
    if '(sub)' in name:
      continue
    # Exception specific to this dataset where a rogue name entry of
    # a single letter 'm' was found
    if name == 'm':
      continue

    similar = process.extract(name, unique_names, limit=5)

    close_matches = []
    for match_name, score in similar:
        # Filter out exact matches, low scores, and second arguments with a slash
        if score >= threshold and match_name != name and '/' not in match_name:
          # Create an order-independent pair
          pair = frozenset([name, match_name])

          # Only add if we haven't seen this exact pair before
          if pair not in seen_pairs and match_name != 'm':
            close_matches.append(match_name)
            seen_pairs.add(pair)

    if close_matches:
      matches[name] = close_matches

  return matches

In [6]:
matches = find_similar_names(X_train)

del matches['Mandeep']
del matches['Hazlewood']
del matches['Roy']
del matches['Mitchell']
del matches['Mitchell Marsh']
matches['Mayank'] = ['Mayank Agarawal']
matches['Rahul'] = ['Rahul Tewatia']

for key, value in matches.items():
  for col in ['striker', 'bowler', 'fielder', 'player_dismissed']:
    X_train.loc[X_train[col].isin(value), col] = key

**Creating the Feature Dataset and Targets for Linear Regression**

In [7]:
# Function to calculate the fantasy points for a player

def match_dreamscore(df:pd.DataFrame):
  scores = defaultdict(int)
  runs = defaultdict(int)
  balls_faced = defaultdict(int)
  conceded = defaultdict(int)
  legal_balls = defaultdict(int)
  wickets = defaultdict(int)
  catches = defaultdict(int)
  out = set()

  for row in df.itertuples(index=False):
    # Striker score
    runs[row.striker] += row.runs_of_bat
    scores[row.striker] += row.runs_of_bat
    if row.runs_of_bat == 4:
      scores[row.striker] += 1
    elif row.runs_of_bat == 6:
      scores[row.striker] += 2
    if not row.wide:
      balls_faced[row.striker] += 1

    # Bowler score
    conceded[row.bowler] += row.runs_of_bat + row.wide + row.noballs
    if not row.wide and not row.noballs:
      legal_balls[row.bowler] += 1

    # Wickets
    if isinstance(row.wicket_type, str):
      wt = row.wicket_type.lower()
      if wt in {'caught', 'bowled', 'lbw', 'stumped', 'hit wicket'}:
        wickets[row.bowler] += 1
        scores[row.bowler] += 25
        if wt in {'bowled', 'lbw'}:
          scores[row.bowler] += 8
      if wt == 'caught' and isinstance(row.fielder, str):
        for f in row.fielder.split('/'):
          catches[f.strip()] += 1
          scores[f.strip()] += 8
      if isinstance(row.player_dismissed, str):
        out.add(row.player_dismissed)

  # Milestones, ducks, strike rate
  for p, r in runs.items():
    if r >= 100:
      scores[p] += 16
    elif r >= 50:
      scores[p] += 8
    elif r >= 30:
      scores[p] += 4
    if r == 0 and p in out and balls_faced[p] > 0:
      scores[p] -= 2

    bf = balls_faced[p]
    if bf >= 10:
      sr = 100 * r / bf
      if sr > 170:
        scores[p] += 6
      elif sr > 150:
        scores[p] += 4
      elif sr >= 130:
        scores[p] += 2
      elif sr < 50:
        scores[p] -= 6
      elif sr < 60:
        scores[p] -= 4
      elif sr <= 70:
        scores[p] -= 2

  # Wicket hauls and economy
  for p, lb in legal_balls.items():
    w = wickets[p]
    if w >= 5:
      scores[p] += 16
    elif w == 4:
      scores[p] += 8
    elif w == 3:
      scores[p] += 4

    if lb >= 12:
      econ = conceded[p] / (lb / 6)
      if econ < 5:
        scores[p] += 6
      elif econ < 6:
        scores[p] += 4
      elif econ <= 7:
        scores[p] += 2
      elif econ > 12:
        scores[p] -= 6
      elif econ > 11:
        scores[p] -= 4
      elif econ >= 10:
        scores[p] -= 2

  # Catch bonus
  for p, c in catches.items():
    if c >= 3:
      scores[p] += 4

  return dict(scores)

In [8]:
# Function to create the statistical features for our linear model

# It takes an input of the player for whom those features are being generated,
# the players playing against him for that match, the names of his team and the opponent team,
# the date of the match, the deliveries dataframe and the home_ground dictionary, which stores
# the home venues for each of the 10 teams.
# 'c' is a constant used for smoothing out the ratio calculations in the features

def create_input(player_name, opposing_players, player_team, opposing_team, date, df, home_ground, c=20.0):
  # Convert dates to datetime objects for accurate chronological filtering
  current_date = pd.to_datetime(date)
  df = df.copy()
  df['date'] = pd.to_datetime(df['date'])

  # Filter for only historical data (matches strictly before the given date)
  past_df = df[df['date'] < current_date].copy()

  # Feature 1: Average venue points (Home Ground & Away Ground)

  # Get the venues for the teams
  home_venue = home_ground.get(player_team)
  away_venue = home_ground.get(opposing_team)

  # Internal function to get the average points scored at that venue, historically
  def get_avg_points_at_venue(venue):
    if not venue:
      return 0
    v_df = past_df[past_df['venue'] == venue]
    if v_df.empty:
      return 0

    match_dates = v_df['date'].unique()
    total_points = 0
    for md in match_dates:
      m_df = v_df[v_df['date'] == md]
      pts = match_dreamscore(m_df)
      total_points += pts.get(player_name, 0)

    return total_points / len(match_dates) if match_dates.size > 0 else 0

  # An equally weighted measure of the average home_venue
  # points and average away_venue points
  avg_home = get_avg_points_at_venue(home_venue)
  avg_away = get_avg_points_at_venue(away_venue)

  f1 = (avg_home + avg_away) / 2.0

  # Feature 2: Match Serial Number Moving Average

  # Determine the match serial number for player_team historically and currently
  team_matches = df[(df['batting_team'] == player_team) | (df['bowling_team'] == player_team)][['season', 'date']].drop_duplicates().sort_values(['season', 'date'])
  team_matches = team_matches.reset_index(drop=True)
  team_matches['match_num'] = team_matches.groupby('season').cumcount() + 1

  # Find the current match serial number (N) and season
  current_match = team_matches[team_matches['date'] == current_date]
  if not current_match.empty:
    curr_season = current_match.iloc[0]['season']
    N = current_match.iloc[0]['match_num']
  else:
    # Fallback if current date is not in DataFrame yet
    curr_season = df[df['date'] <= current_date]['season'].max()
    N = len(team_matches[(team_matches['season'] == curr_season) & (team_matches['date'] < current_date)]) + 1

  target_points = []

  for season, group in team_matches.groupby('season'):
    if season == curr_season:
      # Ongoing season: Previous 3 matches (N-3 to N-1)
      mask = (group['match_num'] >= N - 3) & (group['match_num'] <= N - 1)
    else:
      # Previous seasons: 1 match before, corresponding match, 1 after (N-1, N, N+1)
      mask = (group['match_num'] >= N - 1) & (group['match_num'] <= N + 1)

    # Calculate the points scored on those dates
    selected_dates = group[mask]['date']
    for d in selected_dates:
      if d < current_date:
        m_df = past_df[past_df['date'] == d]
        pts = match_dreamscore(m_df)
        target_points.append(pts.get(player_name, 0))

  f2 = sum(target_points) / len(target_points) if target_points else 0

  # Feature 3: Player vs Opposing Players (Log Ratio)

  player_vs_opp_pts = 0
  opp_vs_player_pts = 0

  for opp in opposing_players:
    # Filter deliveries to exact head-to-head instances (bat vs bowl or fielder involvements)
    mask = ((past_df['striker'] == player_name) & (past_df['bowler'] == opp)) | \
            ((past_df['striker'] == opp) & (past_df['bowler'] == player_name)) | \
            ((past_df['player_dismissed'] == player_name) & (past_df['fielder'].str.contains(opp, na=False, regex=False))) | \
            ((past_df['player_dismissed'] == opp) & (past_df['fielder'].str.contains(player_name, na=False, regex=False)))

    # Calculate the total historical points accordingly
    subset = past_df[mask]
    if not subset.empty:
      pts = match_dreamscore(subset)
      player_vs_opp_pts += pts.get(player_name, 0)
      opp_vs_player_pts += pts.get(opp, 0)

  # Calculate Log ratio with Laplace smoothing 'c'
  # Ratio: Player's points agaisnt opposing players / Opposing players' points against player

  f3 = math.log((player_vs_opp_pts + c) / (opp_vs_player_pts + c))

  # Feature 4: Team vs Team (Log Ratio)

  # Filter historical matches between the two specific teams
  h2h_df = past_df[
      ((past_df['batting_team'] == player_team) & (past_df['bowling_team'] == opposing_team)) |
      ((past_df['batting_team'] == opposing_team) & (past_df['bowling_team'] == player_team))
  ]

  # Map who played for which team during these specific matches
  pt_roster = set(h2h_df[h2h_df['batting_team'] == player_team]['striker']).union(
              set(h2h_df[h2h_df['bowling_team'] == player_team]['bowler']))
  ot_roster = set(h2h_df[h2h_df['batting_team'] == opposing_team]['striker']).union(
              set(h2h_df[h2h_df['bowling_team'] == opposing_team]['bowler']))

  team_pts = 0
  opp_pts = 0

  # Calculate the total historical team points accordingly
  h2h_dates = h2h_df['date'].unique()
  for d in h2h_dates:
    m_df = h2h_df[h2h_df['date'] == d]
    pts = match_dreamscore(m_df)
    for p, score in pts.items():
      if p in pt_roster:
        team_pts += score
      elif p in ot_roster:
        opp_pts += score

  # Ratio: Player's team total points / Opposing team's total points
  f4 = math.log((team_pts + (25*c)) / (opp_pts + (25*c)))

  return [f1, f2, f3, f4]

In [9]:
# Defining the home_ground dictionary

home_ground = {'MI' : 'Wankhede Stadium, Mumbai',
               'PBKS' : 'Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur, Chandigarh',
               'GT' : 'Narendra Modi Stadium, Ahmedabad',
               'RCB' : 'M.Chinnaswamy Stadium, Bengaluru',
               'RR' : 'Sawai Mansingh Stadium, Jaipur',
               'SRH' : 'Rajiv Gandhi International Stadium, Hyderabad',
               'DC' : 'Arun Jaitley Stadium, Delhi',
               'LSG' : 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow',
               'KKR' : 'Eden Gardens, Kolkata',
               'CSK' : 'MA Chidambaram Stadium, Chennai'}

In [10]:
# Creating the new training dataset using the above 4 features

def build_dataset(df: pd.DataFrame, home_ground: dict):
  # Pre-process date column once globally for speed
  df = df.copy()
  df['date'] = pd.to_datetime(df['date'])

  # Sort chronologically
  df = df.sort_values('date').reset_index(drop=True)

  X_list = []
  y_list = []
  metadata = []

  # Group deliveries by unique match (date & venue combination)
  matches = df.groupby(['date', 'venue'])

  for (match_date, venue), match_df in matches:
    # Determine the two competing teams in this match
    teams = match_df['batting_team'].unique()
    if len(teams) < 2:
      continue  # Skip incomplete match records if any

    team_a, team_b = teams[0], teams[1]

    # Calculate actual fantasy points scored in THIS match for target y
    actual_scores = match_dreamscore(match_df)

    # Extract active players in this match for both teams
    def get_team_players(team_name):
      batting = set(match_df[match_df['batting_team'] == team_name]['striker'])
      bowling = set(match_df[match_df['bowling_team'] == team_name]['bowler'])

      # Catch fielders involvement
      fielders = set()
      for f_str in match_df[match_df['bowling_team'] == team_name]['fielder'].dropna():
        fielders.update([f.strip() for f in str(f_str).split('/')])

      return list(batting | bowling | fielders)

    team_a_players = get_team_players(team_a)
    team_b_players = get_team_players(team_b)

    # Process Team A players
    for player in team_a_players:
      features = create_input(
          player_name=player,
          opposing_players=team_b_players,
          player_team=team_a,
          opposing_team=team_b,
          date=match_date,
          df=df,
          home_ground=home_ground
      )
      actual_y = actual_scores.get(player, 0)

      X_list.append(features)
      y_list.append(actual_y)
      metadata.append({'date': match_date, 'player': player, 'team': team_a})

    # Process Team B players
    for player in team_b_players:
      features = create_input(
          player_name=player,
          opposing_players=team_a_players,
          player_team=team_b,
          opposing_team=team_a,
          date=match_date,
          df=df,
          home_ground=home_ground
      )
      actual_y = actual_scores.get(player, 0)

      X_list.append(features)
      y_list.append(actual_y)
      metadata.append({'date': match_date, 'player': player, 'team': team_b})

    print(match_date)

  # Convert to NumPy arrays for scikit-learn
  X = np.array(X_list, dtype=np.float64)
  y = np.array(y_list, dtype=np.float64)
  meta_df = pd.DataFrame(metadata)

  # Clean any NaN/Inf values that might occur from extreme edge log cases
  X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

  return X, y, meta_df

In [11]:
"""
# Generating the features and targets, then saving them
features, target, meta_df = build_dataset(X_train, home_ground)

np.save('X_features', features)
np.save('y_target', target)
np.save('meta_df', meta_df)
"""

"\n# Generating the features and targets, then saving them\nfeatures, target, meta_df = build_dataset(X_train, home_ground)\n\nnp.save('X_features', features)\nnp.save('y_target', target)\nnp.save('meta_df', meta_df)\n"

In [12]:
# Loading the saved features, targets, and metadata
SAVED_DIR = "./Stat_Features/"
features = np.load(SAVED_DIR + 'X_features.npy', allow_pickle=True)
target = np.load(SAVED_DIR + 'y_target.npy', allow_pickle=True)
metadata = np.load(SAVED_DIR + 'meta_df.npy', allow_pickle=True)

In [13]:
# Initialize and fit Linear Regression
model = LinearRegression()
model.fit(features, target)

print("Feature Coefficients:")
feature_names = ['Venue Avg', 'Season Time Moving Average', 'Player-vs-Player Log Ratio', 'Team-vs-Team Log Ratio']
for name, coef in zip(feature_names, model.coef_):
  print(f" - {name}: {coef:.4f}")
print(f" - Intercept: {model.intercept_:.4f}")

Feature Coefficients:
 - Venue Avg: 0.2726
 - Season Time Moving Average: 0.1519
 - Player-vs-Player Log Ratio: 1.4494
 - Team-vs-Team Log Ratio: -7.5671
 - Intercept: 29.9748


In [14]:
# Saving the trained model

with open('Dream11_Team_Regressor.pkl', 'wb') as file:
    pickle.dump(model, file)

#**Model Evaluation**

In [15]:
# Loading the trained model
model = LinearRegression()
with open(SAVED_DIR + 'Dream11_Team_Regressor.pkl', 'rb') as file:
    model = pickle.load(file)

**Loading the Delivery Data for the (2022-2026) Seasons**

In [16]:
DATASET_DIR = 'Datasets/IPL DELIVERIES/'

data = pd.read_csv(DATASET_DIR + 'ipl_2022_deliveries.csv')
for i in [2023,2024,2025,2026]:
    data = pd.concat([data, pd.read_csv(DATASET_DIR + f'ipl_{i}_deliveries.csv')])

df_eval = pd.DataFrame(data)

df_eval['date'] = pd.to_datetime(df_eval['date'], format='%b %d, %Y')

X_eval = df_eval[['season', 'date', 'venue', 'batting_team', 'bowling_team', 'over', 'striker', 'bowler', 'runs_of_bat', 'extras', 'wide', 'legbyes', 'byes', 'noballs', 'wicket_type', 'player_dismissed', 'fielder']]

for key, value in matches.items():
    for col in ['striker', 'bowler', 'fielder', 'player_dismissed']:
        X_eval.loc[X_eval[col].isin(value), col] = key

In [17]:
# Function to define the roles of each loaded player
# during inference
# Is he a batsman, bowler, wicket-keeper or all-rounder?

BAT_MIN_FACED = 75   # >= this -> pure batter
BOWL_MAX_FACED = 25  # <= this -> pure bowler
                     # in between -> all-rounder


def load_roles(path, bat_min=BAT_MIN_FACED, bowl_max=BOWL_MAX_FACED):
  # Reading the player behaviour data file
  b = pd.read_csv(path, index_col=0)
  b.columns = ["name", "pct_faced", "pct_bowled", "is_wk"]

  # Assess the percentages of the types of balls
  # faced by the player against the thresholds:
  def role(r):
    if r.is_wk:
      return "WK"
    if r.pct_faced >= bat_min:
      return "BAT"
    if r.pct_faced <= bowl_max:
      return "BOWL"
    return "AR"

  b["role"] = b.apply(role, axis=1)
  return dict(zip(b["name"], b["role"]))

roles = load_roles('./csv files/player_behaviour_data.csv')

**Loading the Squad Lists for all the Matches from Cricsheet and Aligning Player Names with the Evaluation Dataset**

In [18]:
cricsheet_path = "./Datasets/ipl_json/"

match_list = []

for file_name in os.listdir(cricsheet_path):
  if file_name.endswith('.json'):
    file_path = os.path.join(cricsheet_path, file_name)

    with open(file_path, 'r', encoding='utf-8') as f:
      try:
        data = json.load(f)
        info = data.get('info', {})

        # Extract teams and dates safely
        teams = info.get('teams', [])
        dates = info.get('dates', ['Unknown'])

        # Extract the roster data dictionary
        players_dict = info.get('players', {})

        # Determine names dynamically based on the active match pairing
        team_1_name = teams[0] if len(teams) > 0 else "Unknown Team 1"
        team_2_name = teams[1] if len(teams) > 1 else "Unknown Team 2"

        # Fetch player list mapping for each team
        team_1_roster = players_dict.get(team_1_name, [])
        team_2_roster = players_dict.get(team_2_name, [])

        match_list.append({
            "match_id": file_name.replace('.json', ''),
            "date": dates[0], # Grab the primary scheduled date
            "team_1": team_1_name,
            "team_2": team_2_name,
            "team_1_players": team_1_roster, # Returns a list of players
            "team_2_players": team_2_roster  # Returns a list of players
        })
      except Exception as e:
        print(f"Error parsing file {file_name}: {e}")

# Compile all match outputs into a structured DataFrame
df_cricsheet = pd.DataFrame(match_list)

df_cricsheet['sort_date'] = pd.to_datetime(df_cricsheet['date'])
df_cricsheet = df_cricsheet.sort_values(by='sort_date').drop(columns=['sort_date']).reset_index(drop=True)
df_cricsheet['date'] = pd.to_datetime(df_cricsheet['date'])
df_cricsheet = df_cricsheet[df_cricsheet['date'] >= '2022-01-01']

In [19]:
# Aligning the player names in Cricsheet's data with those in the
# deliveries dataset using the "process" method

def align_cricsheet_names(df_cricsheet, df_original, threshold=90):
  # Explicit exception for the rogue data entry 'm'
  exclude_names = ['m']

  # Gather all unique reference names from the original dataset
  original_cols = ['striker', 'bowler', 'fielder', 'player_dismissed']
  reference_names = set()

  for col in original_cols:
    if col in df_original.columns:
      reference_names.update(df_original[col].dropna().unique())

  for name in exclude_names:
    reference_names.discard(name)
  reference_names_list = list(reference_names)

  # Extract all unique names across team_1_players and team_2_players lists
  cricsheet_names = set()
  for player_list in df_cricsheet['team_1_players'].dropna():
    if isinstance(player_list, list):
      cricsheet_names.update(player_list)

  for player_list in df_cricsheet['team_2_players'].dropna():
    if isinstance(player_list, list):
      cricsheet_names.update(player_list)

  # Build the mapping dictionary (Cricsheet Name -> Original DF Name)
  name_mapping = {}
  low_confidence = {}

  for name in cricsheet_names:
    # Find the single highest-scoring match from original dataset
    best_match, score = process.extractOne(name, reference_names_list)

    if score >= threshold:
      name_mapping[name] = best_match
    else:
      # Keep original name if no match exceeds threshold
      name_mapping[name] = name
      low_confidence[name] = (best_match, score)

  # Replace names inside the list columns of df_cricsheet
  df_cricsheet_mapped = df_cricsheet.copy()

  # Internal function to replace the names
  def replace_names_in_list(players):
    if isinstance(players, list):
      return [name_mapping.get(player, player) for player in players]
    return players

  df_cricsheet_mapped['team_1_players'] = df_cricsheet_mapped['team_1_players'].apply(replace_names_in_list)
  df_cricsheet_mapped['team_2_players'] = df_cricsheet_mapped['team_2_players'].apply(replace_names_in_list)

  # Returning the Cricsheet dataframe with its names aligned and
  # the name mapping dictionary
  return df_cricsheet_mapped, name_mapping, low_confidence

In [20]:
df_cricsheet_mapped, name_mapping, low_confidence = align_cricsheet_names(df_cricsheet, X_eval)
name_mapping['K Yadav'] = 'Kuldeep Yadav'
name_mapping['RA Jadeja'] = 'Ravindra Jadeja'
name_mapping['Mohammad Nabi'] = 'Mohammad Nabi'
name_mapping['Ishan Kishan'] = 'Ishan Kishan'

In [21]:
# Updating the names in the "roles" dictionary to match those
# in the deliveries dataset using the "process" method

def update_roles_dictionary(roles_dict, name_mapping_dict, threshold=90):
  # Extract unique standardized names from the mapping values
  target_names = list(set(name_mapping_dict.values()))

  updated_roles = {}
  unmatched_roles = {}

  for current_name, role in roles_dict.items():
    # Find the best match in our list of standardized names
    best_match, score = process.extractOne(current_name, target_names)

    if score >= threshold:
      # Update key to the matched standardized name
      updated_roles[best_match] = role
    else:
      # Keep original key if no confident match is found
      updated_roles[current_name] = role
      # Track failures to review later
      unmatched_roles[current_name] = (best_match, score)

  return updated_roles, unmatched_roles

In [22]:
updated_roles, missing_roles = update_roles_dictionary(roles, name_mapping)

In [23]:
# Caching the fantasy points data for optimized evaluation/inference

def build_historical_cache(df):
  df = df.copy()
  df['date'] = pd.to_datetime(df['date'])

  # Map: date -> {player_name: fantasy_points}
  match_scores_lookup = {}
  for match_date, m_df in df.groupby('date'):
      match_scores_lookup[match_date] = match_dreamscore(m_df)

  # Map: venue -> [list of match dates at that venue]
  venue_dates_lookup = df.groupby('venue')['date'].unique().to_dict()

  # Pre-calculate team match serial numbers per season globally
  team_df = df[['season', 'date', 'batting_team', 'bowling_team']].drop_duplicates()

  # Unpivot so every row is (season, date, team)
  teams_long = pd.concat([
      team_df[['season', 'date', 'batting_team']].rename(columns={'batting_team': 'team'}),
      team_df[['season', 'date', 'bowling_team']].rename(columns={'bowling_team': 'team'})
  ]).drop_duplicates().sort_values(['team', 'season', 'date']).reset_index(drop=True)

  # Assign 1-indexed serial number per team per season
  teams_long['match_num'] = teams_long.groupby(['team', 'season']).cumcount() + 1

  return match_scores_lookup, venue_dates_lookup, teams_long

# Function to calculate the feature f4 once per team per match
# for optimized evaluation/inference

def compute_team_vs_team_feature(player_team, opposing_team, past_df, match_scores_lookup, c=20.0):
  # Get the dates of past matches
  h2h_dates = past_df[
      ((past_df['batting_team'] == player_team) & (past_df['bowling_team'] == opposing_team)) |
      ((past_df['batting_team'] == opposing_team) & (past_df['bowling_team'] == player_team))
  ]['date'].unique()

  team_pts = 0
  opp_pts = 0

  for d in h2h_dates:
    # Get the past scores from the cache
    m_scores = match_scores_lookup.get(d, {})
    m_df = past_df[past_df['date'] == d]

    pt_roster = set(m_df[m_df['batting_team'] == player_team]['striker']).union(
                set(m_df[m_df['bowling_team'] == player_team]['bowler']))
    ot_roster = set(m_df[m_df['batting_team'] == opposing_team]['striker']).union(
                set(m_df[m_df['bowling_team'] == opposing_team]['bowler']))

    # Calculate the total team scores accordingly
    for p, score in m_scores.items():
      if p in pt_roster:
        team_pts += score
      elif p in ot_roster:
        opp_pts += score

  # Calculate f4
  return math.log((team_pts + (25*c)) / (opp_pts + (25*c)))

# Optimized version of the "create_input" function for faster evaluation/inference
# It makes use of the fantasy points cache and the pre-computed f4 values

def create_input_fast(player_name, opposing_players, player_team, opposing_team,
                               current_date, past_df, home_ground, match_scores_lookup,
                               venue_dates_lookup, teams_long, f4_precomputed, c=20.0):

  # Feature 1: Average venue points

  # Internal function to get the average points scored at that venue, historically
  def get_avg_venue_pts(venue):
    if not venue or venue not in venue_dates_lookup:
      return 0.0
    past_v_dates = [d for d in venue_dates_lookup[venue] if d < current_date]
    if not past_v_dates:
      return 0.0
    return sum(match_scores_lookup[d].get(player_name, 0) for d in past_v_dates) / len(past_v_dates)

  # An equally weighted measure of the average home_venue
  # points and average away_venue points
  avg_home = get_avg_venue_pts(home_ground.get(player_team))
  avg_away = get_avg_venue_pts(home_ground.get(opposing_team))

  f1 = (avg_home + avg_away) / 2.0

  # Feature 2: Match Serial Moving Average

  team_schedule = teams_long[teams_long['team'] == player_team]
  curr_match_row = team_schedule[team_schedule['date'] == current_date]

  if not curr_match_row.empty:
    curr_season = curr_match_row.iloc[0]['season']
    N = curr_match_row.iloc[0]['match_num']
  else:
    past_team = team_schedule[team_schedule['date'] < current_date]
    curr_season = past_team['season'].max() if not past_team.empty else current_date.year
    N = len(past_team[past_team['season'] == curr_season]) + 1

  target_points = []
  for season, group in team_schedule.groupby('season'):
    if season == curr_season:
      # Ongoing season: Previous 3 matches (N-3 to N-1)
      mask = (group['match_num'] >= N - 3) & (group['match_num'] <= N - 1)
    else:
      # Previous seasons: 1 match before, corresponding match, 1 after (N-1, N, N+1)
      mask = (group['match_num'] >= N - 1) & (group['match_num'] <= N + 1)

    # Update the points accordingly
    for d in group[mask]['date']:
      if d < current_date:
        target_points.append(match_scores_lookup[d].get(player_name, 0))

  f2 = sum(target_points) / len(target_points) if target_points else 0.0

  # Feature 3: Player vs Opposing Players (Pre-filtered search space)

  # Getting the records for the specific player
  p_mask = (past_df['striker'] == player_name) | \
            (past_df['bowler'] == player_name) | \
            (past_df['player_dismissed'] == player_name) | \
            (past_df['fielder'].str.contains(player_name, na=False, regex=False))

  p_past_df = past_df[p_mask]

  player_vs_opp_pts = 0
  opp_vs_player_pts = 0

  if not p_past_df.empty:
    for opp in opposing_players:
      # Filter deliveries to exact head-to-head instances (bat vs bowl or fielder involvements)
      opp_mask = ((p_past_df['striker'] == player_name) & (p_past_df['bowler'] == opp)) | \
                  ((p_past_df['striker'] == opp) & (p_past_df['bowler'] == player_name)) | \
                  ((p_past_df['player_dismissed'] == player_name) & (p_past_df['fielder'].str.contains(opp, na=False, regex=False))) | \
                  ((p_past_df['player_dismissed'] == opp) & (p_past_df['fielder'].str.contains(player_name, na=False, regex=False)))

      subset = p_past_df[opp_mask]
      if not subset.empty:
        # Calculate the total points accordingly
        pts = match_dreamscore(subset)
        player_vs_opp_pts += pts.get(player_name, 0)
        opp_vs_player_pts += pts.get(opp, 0)

  # Calculate Log ratio with Laplace smoothing 'c'
  # Ratio: Player's points agaisnt opposing players / Opposing players' points against player

  f3 = math.log((player_vs_opp_pts + c) / (opp_vs_player_pts + c))

  # Feature 4: Use Pre-computed Match Level Value

  f4 = f4_precomputed

  return [f1, f2, f3, f4]

In [24]:
match_scores_lookup, venue_dates_lookup, teams_long = build_historical_cache(X_eval)

In [25]:
# Function to select the Dream 11 Team with the constraints:
# 1-8 Batsmen
# 1-8 Bowlers
# 1-8 Wicket-Keepers
# 1-8 All-Rounders
# At least one player from each team

def select_optimal_xi(players_data):
  N = len(players_data)
  if N < 11:
    return [p['name'] for p in players_data]

  # Objective: Maximize sum(score * x_i) -> Minimize sum(-score * x_i)
  c = [-p['score'] for p in players_data]
  integrality = np.ones(N)  # Force binary selection (0 or 1)
  bounds = [(0, 1) for _ in range(N)]

  A_ub = []
  b_ub = []

  # Constraint: Role bounds (1 to 8 per role)
  for r in ['BAT', 'BOWL', 'WK', 'AR']:
    r_indices = [i for i, p in enumerate(players_data) if p['role'] == r]
    if r_indices:
      # sum(x_i) <= 8
      row_max = [0] * N
      for idx in r_indices: row_max[idx] = 1
      A_ub.append(row_max)
      b_ub.append(8)

      # sum(x_i) >= 1  =>  -sum(x_i) <= -1
      row_min = [0] * N
      for idx in r_indices: row_min[idx] = -1
      A_ub.append(row_min)
      b_ub.append(-1)

  # Constraint: At least 1 player from each team
  teams = list(set(p['team'] for p in players_data))
  for t in teams:
    t_indices = [i for i, p in enumerate(players_data) if p['team'] == t]
    if t_indices:
      row_team = [0] * N
      for idx in t_indices: row_team[idx] = -1
      A_ub.append(row_team)
      b_ub.append(-1)

  # Constraint: Exactly 11 players
  A_eq = [[1] * N]
  b_eq = [11]

  res = linprog(
      c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
      bounds=bounds, integrality=integrality, method='highs'
  )

  if res.success:
    selected_indices = np.where(np.round(res.x) == 1)[0]
    return [players_data[i]['name'] for i in selected_indices]
  else:
    # Fallback greedy selection if constraints cannot be satisfied fully
    sorted_p = sorted(players_data, key=lambda x: x['score'], reverse=True)
    return [p['name'] for p in sorted_p[:11]]

# Evaluation Script for 2026 Season Matches

def evaluate_2026_season(df_deliveries, df_cricsheet_mapped, model, roles, home_ground,
                               match_scores_lookup, venue_dates_lookup, teams_long, c=20.0):

  df_deliveries = df_deliveries.copy()
  df_deliveries['date'] = pd.to_datetime(df_deliveries['date'])

  df_cricsheet_mapped = df_cricsheet_mapped.copy()
  df_cricsheet_mapped['date'] = pd.to_datetime(df_cricsheet_mapped['date'])

  df_2026_matches = df_cricsheet_mapped[df_cricsheet_mapped['date'].dt.year == 2026].sort_values('date').reset_index(drop=True)

  match_overlaps = []
  match_maes = []
  results = []

  print(f"Starting evaluation on {len(df_2026_matches)} matches in 2026:\n")

  for row in df_2026_matches.itertuples(index=False):
    match_date = row.date
    t1, t2 = row.team_1, row.team_2
    squad1, squad2 = row.team_1_players, row.team_2_players

    # Pre-slice past deliveries ONCE per match
    past_df = df_deliveries[df_deliveries['date'] < match_date]

    # Pre-compute Feature 4 ONCE per match for both teams
    f4_t1 = compute_team_vs_team_feature(t1, t2, past_df, match_scores_lookup, c=c)
    f4_t2 = -f4_t1  # log(A/B) = -log(B/A)

    pred_player_pool = []

    # Process Team 1 Squad
    for player in squad1:
      feats = create_input_fast(
          player, squad2, t1, t2, match_date, past_df, home_ground,
          match_scores_lookup, venue_dates_lookup, teams_long, f4_precomputed=f4_t1, c=c
      )
      pred_score = model.predict([feats])[0]
      pred_player_pool.append({
          'name': player, 'score': pred_score,
          'role': roles.get(player, 'BAT'), 'team': t1
      })

    # Process Team 2 Squad
    for player in squad2:
      feats = create_input_fast(
          player, squad1, t2, t1, match_date, past_df, home_ground,
          match_scores_lookup, venue_dates_lookup, teams_long, f4_precomputed=f4_t2, c=c
      )
      pred_score = model.predict([feats])[0]
      pred_player_pool.append({
          'name': player, 'score': pred_score,
          'role': roles.get(player, 'BAT'), 'team': t2
      })

    # Predict Best XI
    predicted_xi = select_optimal_xi(pred_player_pool)

    # Fetch Actual Scores directly from O(1) Cache
    actual_scores = match_scores_lookup.get(match_date, {})

    actual_player_pool = [
        {'name': p['name'], 'score': actual_scores.get(p['name'], 0), 'role': p['role'], 'team': p['team']}
        for p in pred_player_pool
    ]

    actual_xi = select_optimal_xi(actual_player_pool)

    # Compute Metrics
    overlap_count = len(set(predicted_xi) & set(actual_xi))
    actual_pts_pred_xi = sum(actual_scores.get(p, 0) for p in predicted_xi)
    actual_pts_act_xi = sum(actual_scores.get(p, 0) for p in actual_xi)
    mae = abs(actual_pts_pred_xi - actual_pts_act_xi)

    match_overlaps.append(overlap_count)
    match_maes.append(mae)

    results.append({
        'date': match_date.strftime('%Y-%m-%d'),
        'teams': f"{t1} vs {t2}",
        'overlap_out_of_11': overlap_count,
        'pred_xi_pts': actual_pts_pred_xi,
        'actual_xi_pts': actual_pts_act_xi,
        'mae': mae
    })

    print(match_date)
    # Overlap count
    print(f"Overlap: {overlap_count} / 11\n")

  avg_overlap = np.mean(match_overlaps) if match_overlaps else 0
  avg_mae = np.mean(match_maes) if match_maes else 0

  print(f"Average Matched Players : {avg_overlap:.2f} / 11")
  print(f"Average Fantasy Total MAE: {avg_mae:.2f} points")

  return pd.DataFrame(results), avg_overlap, avg_mae

In [26]:
# Run Evaluation
eval_df, avg_matches, avg_mae = evaluate_2026_season(
    df_deliveries=X_eval,
    df_cricsheet_mapped=df_cricsheet_mapped,
    model=model,
    roles=updated_roles,
    home_ground=home_ground,
    match_scores_lookup=match_scores_lookup,
    venue_dates_lookup=venue_dates_lookup,
    teams_long=teams_long,
    c=20.0
)

Starting evaluation on 74 matches in 2026:

2026-03-28 00:00:00
Overlap: 9 / 11

2026-03-29 00:00:00
Overlap: 5 / 11

2026-03-30 00:00:00
Overlap: 8 / 11

2026-03-31 00:00:00
Overlap: 6 / 11

2026-04-01 00:00:00
Overlap: 3 / 11

2026-04-02 00:00:00
Overlap: 7 / 11

2026-04-03 00:00:00
Overlap: 8 / 11

2026-04-04 00:00:00
Overlap: 6 / 11

2026-04-04 00:00:00
Overlap: 8 / 11

2026-04-05 00:00:00
Overlap: 9 / 11

2026-04-05 00:00:00
Overlap: 5 / 11

2026-04-06 00:00:00
Overlap: 4 / 11

2026-04-07 00:00:00
Overlap: 6 / 11

2026-04-08 00:00:00
Overlap: 6 / 11

2026-04-09 00:00:00
Overlap: 8 / 11

2026-04-10 00:00:00
Overlap: 8 / 11

2026-04-11 00:00:00
Overlap: 6 / 11

2026-04-11 00:00:00
Overlap: 7 / 11

2026-04-12 00:00:00
Overlap: 7 / 11

2026-04-12 00:00:00
Overlap: 9 / 11

2026-04-13 00:00:00
Overlap: 7 / 11

2026-04-14 00:00:00
Overlap: 8 / 11

2026-04-15 00:00:00
Overlap: 7 / 11

2026-04-16 00:00:00
Overlap: 5 / 11

2026-04-17 00:00:00
Overlap: 7 / 11

2026-04-18 00:00:00
Overlap: 8 